In [ ]:
from dotenv import load_dotenv
from fastmcp import Client
from openai import OpenAI
from pathlib import Path
import os
import json

load_dotenv(dotenv_path=Path("./../.env"))

BASE_URL = os.getenv("BASE_URL")
MCP_URL = os.getenv("MCP_URL")
MODEL = os.getenv("MODEL")

if any(x is None for x in [BASE_URL, MCP_URL, MODEL]):
    print("Env is None")
    exit()


SYSTEM_PROMPT="""
You are a movie information assistant designed to help users discover, learn about, and discuss films.

Guidelines:
- Use available tools proactively when queries require current information, data verification, or external sources
- If a tool returns insufficient results, refine your query and retry once before acknowledging limitations
- If a user's query appears to contain spelling or logical errors, clarify rather than silently correct—ask before assuming
- Prioritize accuracy: verify results are adequate before responding; if gaps remain, use additional tools
- Provide natural, transparent responses; avoid mentioning tool mechanics or implementation details
- Keep responses concise while ensuring completeness and clarity
- When uncertain, acknowledge limitations honestly rather than speculating

Be concise but complete."""

llm = OpenAI(
    base_url=BASE_URL,
    api_key="none"
)

async def get_tools(mcp: Client):
    raw = await mcp.list_tools()

    return [
        {
            "type": "function",
            "function": {
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema or {
                    "type": "object",
                    "properties": {}
                }
            }
        }
        for t in raw
    ]

async def agent(user_message: str):
    mcp = Client(MCP_URL)  #type: ignore
    async with mcp:
        tools = await get_tools(mcp)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message}
        ]

        while True:
            response = llm.chat.completions.create(
                model=MODEL, #type: ignore
                messages=messages, #type: ignore
                tools=tools, #type: ignore
                tool_choice="auto"
            )

            msg = response.choices[0].message

            if not msg.tool_calls:
                return msg.content

            messages.append(msg) #type: ignore

            for tc in msg.tool_calls:

                name = tc.function.name #type: ignore
                
                args = json.loads(tc.function.arguments) #type: ignore
                result = await mcp.call_tool(name, args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": str(result.content)
                })

In [46]:
res = await agent("What are the ratings of movies Bambi 2?")
print(res)

I couldn't find any ratings for *Bambi II* across major movie review sites like IMDb, Rotten Tomatoes, or Metacritic. It appears to be a very obscure film with limited critical reception.


In [ ]:
res = await agent("What question I asked before?")
print(res)

I'm sorry, but as a new conversation, I don’t have memory of previous turns. Therefore, I do not know what question you previously asked.


In [48]:
res = await agent( "How many individual bunny characters intended to have the 'Bambi' film?")
print(res)

Originally, the film was intended to have six individual bunny characters, similar to the dwarfs in Snow White. However, the concept evolved to include five generic rabbits and one rabbit with a distinct color and feature (like one tooth). The original cast included characters like Thumper, Flower, and Faline.


In [49]:
res = await agent( "Get the summary of movie Bambi!")
print(res)

Bambi is the story of a young white-tailed deer named Bambi who grows up in a forest. He befriends Thumper and Flower, learns about the world from his mother, and eventually becomes the Great Prince of the Forest after facing challenges like losing his mother to a hunter and surviving a devastating forest fire.


In [50]:
res = await agent( "Access the resources from the mcp. What you have there?")
print(res)

Okay, I see a list of Disney movies available: Alice in Wonderland, Bambi, Cinderella, Dumbo, Pinocchio, Snow White and the Seven Dwarfs, The Jungle Book, The Lion King, The Tigger Movie, and Toy Story.  How can I help you with any of these or perhaps another movie?


In [51]:
res = await agent( "What movies you have available?")
print(res)

I have the following movies available: Alice in Wonderland, Bambi, Cinderella, Dumbo, Pinocchio, Snow White and the Seven Dwarfs, The Jungle Book, The Lion King, The Tigger Movie, and Toy Story. Would you like to know more about any of them?
